# 01 Descarga, carga y limpieza del grafo

**Proyecto:** accesibilidad peatonal a equipamiento urbano – Miraflores + San Juan de Miraflores (Hito 1)

Este notebook cubre los pasos 2 a 4 del plan (`docs/hito1`):

- **Paso 2:** descarga de datos y respaldo reproducible
- Paso 3: carga e inventario
- Paso 4: limpieza y proyección

## Paso 2 – Descarga de datos

### 2.1 Configuración, semilla y versiones

In [ ]:
import sys, json, hashlib, platform, datetime, time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import shapely
import osmnx as ox
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from src import config as cfg

np.random.seed(cfg.SEED)
ox.settings.timeout = 300
ox.settings.requests_timeout = 60   # un intento colgado no bloquea 180 s; se reintenta abajo
ox.settings.use_cache = True
ox.settings.log_console = False

for d in (cfg.DATA_RAW, cfg.DATA_INTERIM, cfg.DATA_PROCESSED):
    d.mkdir(parents=True, exist_ok=True)

versiones = {
    "python": platform.python_version(),
    "osmnx": ox.__version__,
    "networkx": nx.__version__,
    "geopandas": gpd.__version__,
    "shapely": shapely.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
versiones

### 2.2 Límites de los distritos

In [ ]:
limites = ox.geocode_to_gdf(cfg.PLACES)
limites["distrito"] = cfg.DISTRICT_LABELS
limites_utm = limites.to_crs(cfg.CRS_METRIC)
limites_utm["area_km2"] = limites_utm.geometry.area / 1e6
limites_utm[["distrito", "area_km2"]].round(2)

### 2.3 Polígono de descarga

Se unen ambos distritos y se agrega un buffer de `BUFFER_M` metros (calculado en UTM, nunca en grados). Así las rutas cercanas al límite no quedan truncadas; en la limpieza se recorta lo que quede fuera de los distritos.

> **Los distritos no son contiguos** (Surco queda entre ambos), por lo que el polígono es un `MultiPolygon` de dos partes y el grafo tendrá al menos dos componentes. Por eso la descarga usa `retain_all=True`: con el valor por defecto (`False`) osmnx conserva solo el componente más grande y descarta el otro distrito completo.

In [ ]:
union_utm = limites_utm.geometry.union_all()
poligono_descarga_utm = union_utm.buffer(cfg.BUFFER_M)
poligono_descarga = gpd.GeoSeries([poligono_descarga_utm], crs=cfg.CRS_METRIC).to_crs(4326).iloc[0]

print(f"Área de los distritos unidos: {union_utm.area/1e6:.2f} km²")
print(f"Área con buffer de {cfg.BUFFER_M} m: {poligono_descarga_utm.area/1e6:.2f} km²")

ax = limites_utm.plot(column="distrito", alpha=0.6, edgecolor="k", legend=True, figsize=(6, 6))
gpd.GeoSeries([poligono_descarga_utm], crs=cfg.CRS_METRIC).boundary.plot(ax=ax, color="r", ls="--")
ax.set_title("Distritos y polígono de descarga (buffer)")
plt.show()

### 2.4 Descarga del grafo peatonal

In [ ]:
# Overpass a veces tarda en conectar; se reintenta con pausa
for intento in range(1, 13):
    try:
        momento = datetime.datetime.now().astimezone()
        G_raw = ox.graph_from_polygon(poligono_descarga, network_type=cfg.NETWORK_TYPE, simplify=True, retain_all=True)
        break
    except Exception as e:
        print(f"Intento {intento} fallido: {type(e).__name__}")
        if intento == 12:
            raise
        time.sleep(5)
print(f"Descargado: {G_raw.number_of_nodes()} nodos, {G_raw.number_of_edges()} aristas")

### 2.4b Verificación de cobertura

Se comprueba que el grafo contiene nodos de **ambos** distritos antes de guardarlo, y se registra el número de componentes conexos.

In [ ]:
nodos_g = ox.graph_to_gdfs(G_raw, edges=False).to_crs(cfg.CRS_METRIC)
etiq = gpd.sjoin(nodos_g[["geometry"]], limites_utm[["distrito", "geometry"]], how="left", predicate="within")
etiq = etiq[~etiq.index.duplicated(keep="first")]["distrito"].fillna("Fuera (buffer)")
nodos_por_distrito = etiq.value_counts().to_dict()
n_componentes = nx.number_weakly_connected_components(G_raw)

print("Nodos por distrito:", nodos_por_distrito)
print("Componentes conexos (débiles):", n_componentes)
faltan = [d for d in cfg.DISTRICT_LABELS if nodos_por_distrito.get(d, 0) == 0]
assert not faltan, f"El grafo no contiene nodos de: {faltan}"

### 2.5 Respaldo en disco y registro de reproducibilidad

Se guarda el grafo crudo (GraphML), los límites (GeoPackage) y un log con fecha, consulta, versiones, conteos y hash SHA-256.

In [ ]:
ruta_grafo = cfg.DATA_RAW / "walk_graph_raw.graphml"
ruta_limites = cfg.DATA_RAW / "limites_distritos.gpkg"
ruta_log = cfg.DATA_RAW / "descarga_log.json"

ox.save_graphml(G_raw, ruta_grafo)
limites[["distrito", "geometry"]].to_file(ruta_limites, driver="GPKG")

def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

log = {
    "fecha_descarga": momento.isoformat(timespec="seconds"),
    "consulta": {
        "funcion": "ox.graph_from_polygon",
        "lugares": cfg.PLACES,
        "network_type": cfg.NETWORK_TYPE,
        "simplify": True,
        "retain_all": True,
        "buffer_m": cfg.BUFFER_M,
        "crs_buffer": cfg.CRS_METRIC,
    },
    "versiones": versiones,
    "grafo_crudo": {
        "archivo": ruta_grafo.name,
        "nodos": G_raw.number_of_nodes(),
        "aristas": G_raw.number_of_edges(),
        "componentes_debiles": n_componentes,
        "nodos_por_distrito": nodos_por_distrito,
        "sha256": sha256(ruta_grafo),
    },
    "limites": {"archivo": ruta_limites.name, "sha256": sha256(ruta_limites)},
}
ruta_log.write_text(json.dumps(log, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(log, indent=2, ensure_ascii=False))